# Cp + Regroup - Separate Container Triangles

Two-step pipeline:

1. **Cp** - run the standard pressure-coefficient transform on the wind-tunnel body / probe H5s and write `cp.default.time_series.h5`.
2. **Regroup** - apply `cfdmod.regroup` with a `ByConnectivityGrouping` to detect each container as a connected component, then emit a new geometry + Cp timeseries where each container is its own labelled surface and triangles are reordered per container.

Default aggregation is `per_triangle` (one HDF5 column per parent triangle, reordered so each container is a contiguous block). Switch to `area_weighted_mean` in the regroup config cell if you want one aggregated value per container instead.

## 1. Imports

In [ ]:
import pathlib

import h5py
import numpy as np
from lnas import LnasFormat

from cfdmod import (
    ByConnectivityGrouping,
    BySizeRoundedPerComponent,
    CpCaseConfig,
    RegroupConfig,
    load_mesh,
    mesh_from_h5,
    run_cp,
    run_regroup,
)
from cfdmod.pressure.parameters import CpConfig

: 

## 2. Configure inputs

Set `DATA_DIR` to wherever the simulation results live. The cell auto-discovers the body H5 (`bodies.*.h5`) and probe H5 (`points.*.h5`) inside it; override the explicit paths below if your filenames differ.

All outputs land in `OUTPUT_DIR` (defaults to `<DATA_DIR>/output_regroup`).

In [ ]:
# --- File paths ----------------------------------------------------------
DATA_DIR = pathlib.Path.home() / "Documents/sim_results/results_container/SN"
OUTPUT_DIR = DATA_DIR / "output_regroup"

# Auto-discover body / probe H5s by the standard naming convention. Override
# explicitly if your filenames differ.
_bodies = sorted(DATA_DIR.glob("bodies.*.h5"))
_probes = sorted(DATA_DIR.glob("points.*.h5"))
BODY_H5 = _bodies[0] if _bodies else None
PROBE_H5 = _probes[0] if _probes else None

# Optional: a fixed-frame reference mesh (.lnas / .stl / .h5 / .xdmf). Leave
# as None to use the body H5's own embedded geometry.
REFERENCE_MESH = None

# --- Cp time window (None = use the body H5's full extent) ---------------
T_MIN = None
T_MAX = None

# --- Cp physics ----------------------------------------------------------
# Required by CpConfig. SIMUL_U_H is the simulation flow velocity used as
# the dynamic-pressure denominator; SIMUL_CHARACTERISTIC_LENGTH is the
# reference length (only used as the L/U time scale when normalize_time
# is True). FLUID_DENSITY defaults to 1 (LBM lattice units).
SIMUL_U_H = 1.0
SIMUL_CHARACTERISTIC_LENGTH = 1.0
FLUID_DENSITY = 1.0
MACROSCOPIC_TYPE = "pressure"  # 'pressure' or 'rho' (LBM density)
REFERENCE_PRESSURE = "probe"  # 'probe' (first probe point) or 'average'

# --- Regroup options -----------------------------------------------------
# Step 1: connectivity split. Triangle clusters smaller than MIN_TRIS are
# dropped (typically loose stray triangles).
MIN_TRIS = 4

# Step 2: subdivide each connected component by target container dimensions.
# Each component is split into max(1, round(extent / target)) cells per axis,
# anchored at that component's own bounding box. Standard 20ft container
# defaults: x=6.34m (length), y=2.58m (width), z=2.6m (height). A single
# container becomes one cell; a row of 5 touching containers becomes 5.
TARGET_SIZE_X = 6.34
TARGET_SIZE_Y = 2.58
TARGET_SIZE_Z = 2.6

# 'sliced'             : geometric 90-degree cuts at every cell boundary
#                        (Ce-style); output triangles never straddle two
#                        cells. Per-fragment HDF5 column inherits its parent
#                        triangle's value. Best for clean per-container
#                        rendering and downstream analysis.
# 'per_triangle'       : no slicing; one HDF5 column per parent triangle,
#                        reordered so each container is a contiguous block.
#                        Faster (no geometric cuts), but boundary triangles
#                        get assigned wholly to one cell by their centroid.
# 'area_weighted_mean' : one aggregated value per container (broadcast).
AGGREGATION = "sliced"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR : {DATA_DIR}")
print(f"BODY_H5  : {BODY_H5}")
print(f"PROBE_H5 : {PROBE_H5}")
print(f"OUTPUT   : {OUTPUT_DIR}")
assert (
    BODY_H5 is not None and BODY_H5.exists()
), f"No bodies.*.h5 found in {DATA_DIR}; set BODY_H5 explicitly above."
assert (
    PROBE_H5 is not None and PROBE_H5.exists()
), f"No points.*.h5 found in {DATA_DIR}; set PROBE_H5 explicitly above."

## 3. Inspect the mesh

Quick sanity check: triangle count and bounding box of the geometry the rest of the notebook will reason about.

In [ ]:
if REFERENCE_MESH is not None:
    mesh = load_mesh(REFERENCE_MESH)
    mesh_source = REFERENCE_MESH
else:
    mesh = mesh_from_h5(BODY_H5)
    mesh_source = BODY_H5
verts = mesh.geometry.vertices

print(f"Mesh source : {mesh_source}")
print(f"Triangles   : {len(mesh.geometry.triangles):,}")
print(f"Vertices    : {len(verts):,}")
print(
    f"Bounding box: "
    f"x=[{verts[:,0].min():.2f}, {verts[:,0].max():.2f}], "
    f"y=[{verts[:,1].min():.2f}, {verts[:,1].max():.2f}], "
    f"z=[{verts[:,2].min():.2f}, {verts[:,2].max():.2f}]"
)

## 4. Build the Cp config

Single body covering every surface in the mesh. We leave `statistics` unset (defaults to `[]`), which skips the stats step entirely - this notebook only needs the per-triangle Cp timeseries that regroup consumes downstream. To also write a `stats.h5`, pass e.g. `statistics=[BasicStatisticModel(stats="mean"), BasicStatisticModel(stats="rms")]` (note the field name is `stats`, plural).


In [ ]:
# Resolve the time window against the body H5's actual extent.
def _detect_time_range(body_h5: pathlib.Path) -> tuple[float, float]:
    with h5py.File(body_h5, "r") as f:
        keys = list(f["pressure"].keys())
    times = sorted(float(k[1:]) for k in keys)
    return float(times[0]), float(times[-1])


_detected = _detect_time_range(BODY_H5)
timestep_range = (
    _detected[0] if T_MIN is None else max(float(T_MIN), _detected[0]),
    _detected[1] if T_MAX is None else min(float(T_MAX), _detected[1]),
)
print(f"Body H5 time range : [{_detected[0]:.3f}, {_detected[1]:.3f}]")
print(f"Cropped to         : [{timestep_range[0]:.3f}, {timestep_range[1]:.3f}]")

cp_cfg = CpCaseConfig(
    pressure_coefficient={
        "default": CpConfig(
            timestep_range=timestep_range,
            simul_U_H=SIMUL_U_H,
            simul_characteristic_length=SIMUL_CHARACTERISTIC_LENGTH,
            fluid_density=FLUID_DENSITY,
            macroscopic_type=MACROSCOPIC_TYPE,
            reference_pressure=REFERENCE_PRESSURE,
        )
    }
)

## 5. Run Cp

Streams the body / probe H5s and writes `cp.default.time_series.{h5,xdmf}`. With `statistics=[]` (the default) no `stats.h5` is produced; add stats to the Cp config above if you also want one.


In [ ]:
run_cp(
    body_h5=BODY_H5,
    probe_h5=PROBE_H5,
    cfg_path=cp_cfg,
    output=OUTPUT_DIR,
    mesh_path=REFERENCE_MESH,
)
CP_TIMESERIES = OUTPUT_DIR / "cp.default.time_series.h5"
print(f"Cp timeseries written to {CP_TIMESERIES}")

## 6. Regroup + per-face aggregate + coarsen (single in-memory pipeline)

All three steps run in memory off the original Cp timeseries -- the 300k+-triangle sliced and per-face intermediates never touch disk.

1. **`ByConnectivityGrouping`** + **`BySizeRoundedPerComponent`**: label each connected component as a parent, then subdivide by container dimensions. The connectivity parents are scaffolding; only the leaf cells (one per container) survive.
2. **Sliced regroup** (in memory): cut triangles along axis-aligned cell boundaries so each fragment lies in exactly one cell. Yields a 346k-fragment mesh + a `regroup_index.new_to_parent` map back to the original `n_parent` triangles.
3. **Per-face bucketing**: quantise each fragment's normal to one of 6 axis-aligned directions (`xp`/`xn`/`yp`/`yn`/`zp`/`zn`) and group by `(container_cell, direction)` -- one bucket per container face. Total area per bucket goes into the area-weighted mean weights.
4. **Coarsen with `remesh_per_group`** (default settings = exact coplanar-fan merge, lossless): each per-face surface collapses to its minimum triangulation (typically 2 triangles per face). Surface names are preserved one-to-one.
5. **Stream Cp per timestep** from `CP_TIMESERIES`: for each `t{T}` we compute the per-face area-weighted mean from the *parent* values + fragment areas (no need to materialise the sliced or per-face HDF5), broadcast to the coarse triangles, and write a single row.

Outputs (only these land on disk):

- `regroup_out/geometry.per_face.remeshed.lnas` -- coarse mesh, one named surface per `(container, face)`.
- `regroup_out/cp.per_face.remeshed.{h5,xdmf}` -- per-face Cp animation on the coarse mesh.


In [ ]:
from cfdmod.regroup import build_regroup_mapping, expand_regroup_chain
from cfdmod.regroup.functions import build_sliced_regrouped_mesh
from cfdmod.geometry.grouping import GroupingResult
from cfdmod.remesh import remesh_per_group
from cfdmod.io.xdmf import write_temporal_xdmf

# --- Configure the regroup chain (sliced aggregation) --------------------
regroup_cfg = RegroupConfig(
    groupings=[
        ByConnectivityGrouping(
            name_template="cc{idx}",
            min_triangles=MIN_TRIS,
        ),
        BySizeRoundedPerComponent(
            target_size_x=TARGET_SIZE_X,
            target_size_y=TARGET_SIZE_Y,
            target_size_z=TARGET_SIZE_Z,
            name_template="{parent}_c{idx}",
        ),
    ],
    aggregation=AGGREGATION,
    timeseries_group="cp",
    output_geometry_format="lnas",
    unassigned_policy="drop",
)

# --- Slice in memory (no disk writes) ------------------------------------
expanded, consumed, parent_intervals, parent_triangles = expand_regroup_chain(
    regroup_cfg.groupings, mesh, regroup_cfg.transformation
)
grouping = build_regroup_mapping(mesh, expanded, regroup_cfg.transformation)
if consumed:
    grouping = GroupingResult(
        parent_n_triangles=grouping.parent_n_triangles,
        groups={n: i for n, i in grouping.groups.items() if n not in consumed},
    )
sliced_lnas, regroup_index = build_sliced_regrouped_mesh(
    mesh,
    grouping,
    parent_intervals=parent_intervals,
    parent_triangles=parent_triangles,
    unassigned_policy="drop",
)
n_frag = sliced_lnas.geometry.triangles.shape[0]
print(f"Sliced (in memory): {n_frag:,} fragments / " f"{len(sliced_lnas.surfaces):,} containers")

# --- Per-face buckets on the sliced mesh ---------------------------------
parent_of = regroup_index.new_to_parent
tri_v = sliced_lnas.geometry.triangle_vertices
crosses = np.cross(tri_v[:, 1] - tri_v[:, 0], tri_v[:, 2] - tri_v[:, 0])
fragment_area = np.linalg.norm(crosses, axis=1) / 2
norms = crosses / (np.linalg.norm(crosses, axis=1, keepdims=True) + 1e-30)
axis_idx = np.abs(norms).argmax(axis=1)
sign_neg = norms[np.arange(n_frag), axis_idx] < 0
direction = (axis_idx * 2 + sign_neg.astype(np.int64)).astype(np.int64)

cell_names = sorted(sliced_lnas.surfaces.keys())
cell_of = np.empty(n_frag, dtype=np.int64)
for ci, name in enumerate(cell_names):
    cell_of[sliced_lnas.surfaces[name]] = ci

group_key = cell_of * 6 + direction
unique_groups, bucket_of = np.unique(group_key, return_inverse=True)
n_buckets = len(unique_groups)
total_area_per_bucket = np.bincount(bucket_of, weights=fragment_area, minlength=n_buckets)
# Guard zero-area buckets (degenerate slicing artifacts on cell boundaries).
safe_area = np.where(total_area_per_bucket > 0, total_area_per_bucket, 1.0)
print(f"Per-face buckets: {n_buckets:,}  ({len(cell_names):,} containers x up to 6 faces)")

# --- Build a per-face named LnasFormat (no disk write) -------------------
direction_suffix = {0: "xp", 1: "xn", 2: "yp", 3: "yn", 4: "zp", 5: "zn"}
per_face_surfaces = {}
for gi, gkey in enumerate(unique_groups):
    name = f"{cell_names[gkey // 6]}_{direction_suffix[gkey % 6]}"
    per_face_surfaces[name] = np.flatnonzero(bucket_of == gi).astype(np.int32)
per_face_lnas = LnasFormat(
    version=sliced_lnas.version,
    geometry=sliced_lnas.geometry,
    surfaces=per_face_surfaces,
)

# --- Coarsen (defaults: exact coplanar merge, lossless) ------------------
remeshed_lnas = remesh_per_group(per_face_lnas)
n_out = remeshed_lnas.geometry.triangles.shape[0]
print(
    f"Remesh: {n_frag:,} fragments -> {n_out:,} coarse triangles "
    f"({(1 - n_out / n_frag) * 100:.1f}% reduction)"
)

# Map each coarse output triangle back to its (cell, face) bucket.
output_tri_bucket = np.empty(n_out, dtype=np.int64)
bucket_for_name = {
    f"{cell_names[g // 6]}_{direction_suffix[g % 6]}": gi for gi, g in enumerate(unique_groups)
}
for name, idxs in remeshed_lnas.surfaces.items():
    if idxs.size:
        output_tri_bucket[idxs] = bucket_for_name[name]

# --- Write the coarse outputs --------------------------------------------
REGROUP_DIR = OUTPUT_DIR / "regroup_out"
REGROUP_DIR.mkdir(parents=True, exist_ok=True)
REMESHED_LNAS = REGROUP_DIR / "geometry.per_face.remeshed.lnas"
REMESHED_H5 = REGROUP_DIR / "cp.per_face.remeshed.h5"
REMESHED_XDMF = REGROUP_DIR / "cp.per_face.remeshed.xdmf"
remeshed_lnas.to_file(REMESHED_LNAS)

# Stream Cp: parent values -> per-face area-weighted mean -> coarse tris.
with h5py.File(CP_TIMESERIES, "r") as src, h5py.File(REMESHED_H5, "w") as dst:
    dst.create_dataset("Triangles", data=remeshed_lnas.geometry.triangles.astype(np.int32))
    dst.create_dataset("Geometry", data=remeshed_lnas.geometry.vertices.astype(np.float64))
    meta_dst = dst.create_group("meta")
    for k in ("time_steps", "time_normalized"):
        meta_dst.create_dataset(k, data=src["meta"][k][:])
    cp_dst = dst.create_group("cp")
    keys = sorted(k for k in src["cp"].keys() if k.startswith("t"))
    for key in keys:
        parent_cp = src["cp"][key][:]
        weighted = fragment_area * parent_cp[parent_of]
        bucket_sum = np.bincount(bucket_of, weights=weighted, minlength=n_buckets)
        bucket_cp = bucket_sum / safe_area
        cp_dst.create_dataset(key, data=bucket_cp[output_tri_bucket].astype(np.float64))

write_temporal_xdmf(REMESHED_H5, REMESHED_XDMF, "cp")
print(f"\nWrote: {REMESHED_LNAS.name}, {REMESHED_H5.name}, {REMESHED_XDMF.name}")
print("No intermediate 300k+-triangle h5/lnas was written.")

## 7. Inspect the coarse output

Per-face surface counts, total triangles, and a sampled summary. Each surface is named `<container>_<direction>` and typically holds two triangles (one rectangular face -> minimum triangulation).


In [ ]:
print(f"Coarse mesh: {remeshed_lnas.geometry.triangles.shape[0]:,} triangles")
print(f"Surfaces   : {len(remeshed_lnas.surfaces):,} (one per container face)")
sizes = np.array([idxs.size for idxs in remeshed_lnas.surfaces.values()])
print(
    f"Per-surface tri counts: min={int(sizes.min())} "
    f"median={int(np.median(sizes))} max={int(sizes.max())}"
)
print("\nFirst 5 surfaces (name, tri count, bounding box):")
tri_verts_out = remeshed_lnas.geometry.triangle_vertices
for name in sorted(remeshed_lnas.surfaces.keys())[:5]:
    idxs = remeshed_lnas.surfaces[name]
    if idxs.size == 0:
        continue
    sub = tri_verts_out[idxs]
    lo = sub.reshape(-1, 3).min(axis=0)
    hi = sub.reshape(-1, 3).max(axis=0)
    print(
        f"  {name:<24s} {idxs.size:>3d} tris  "
        f"x=[{lo[0]:7.2f}, {hi[0]:7.2f}]  "
        f"y=[{lo[1]:7.2f}, {hi[1]:7.2f}]  "
        f"z=[{lo[2]:7.2f}, {hi[2]:7.2f}]"
    )

## 8. Open in ParaView

| File | What it shows |
|---|---|
| `output_regroup/cp.default.time_series.xdmf` | The original Cp animation on the full unsegmented mesh. |
| `output_regroup/regroup_out/cp.per_face.remeshed.xdmf` | The per-face Cp animation on the coarse mesh -- one named surface per `(container, face)`. Use ParaView's **Extract Block** filter to isolate a single face (`<cell>_zp`, `<cell>_xn`, ...). |
